<a href="https://colab.research.google.com/github/samiraghafarigousheh-sys/PyBuildingEnergy_AIBteam_AU/blob/main/copy_of_pybuildingenergy_my_house.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
from google.colab import drive
drive.mount("/content/drive")

ValueError: Mountpoint must not already contain files

In [24]:
import os
folder_path='/content/drive/MyDrive/pybuildingenergy_myhouse'
os.makedirs(folder_path, exist_ok=True)


In [43]:
!git clone https://github.com/EURAC-EEBgroup/pyBuildingEnergy.git
%cd pyBuildingEnergy

fatal: destination path 'pyBuildingEnergy' already exists and is not an empty directory.
/content/pybuildinenergy_AIB/pybuildinenergy_AIB/pybuildinenergy_AIB/pyBuildingEnergy


In [44]:
!pip install pybuildingenergy
import numpy as np
import pandas as pd
import os
import pybuildingenergy as pybui
from pybuildingenergy.source.check_input import sanitize_and_validate_BUI
from pybuildingenergy.source.graphs import Graphs_and_report
from pybuildingenergy.source.utils import ISO52016

In [45]:
#1 Configuration
weather_file   = None                                   # use PVGIS (API-based, no local file needed)
WEATHER_SOURCE = "pvgis"                                # latest PVGIS 2016-2023 TMY for Melbourne

OUTPUT_DIR = folder_path
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [46]:
 #  3) CONSTRUCTION U-VALUES & THERMAL CAPACITY

# U-values (W / m²·K) — Australian BCA 2006 minimum-spec
U_EXT_WALL  = 1.00   # brick veneer / precast w/ R1.0 insulation
U_INT_WALL  = 2.50   # concrete block + plasterboard, no insulation
U_INT_SLAB  = 1.80   # 200 mm concrete intermediate floor
U_WINDOW    = 5.40   # aluminium-frame single glazing
G_WINDOW    = 0.65   # SHGC of clear single glazing

# Solar absorptance — DARK RED BRICK (confirmed from photo)
ABS_EXT_WALL = 0.75  # dark red brick (was 0.55 for mid-tone, corrected after seeing photo)
ABS_INT      = 0.0   # interior surfaces

# Areal thermal capacity (J / m²·K)
C_EXT_WALL = 450_000   # heavy concrete external wall
C_INT_WALL = 330_000   # concrete-block partition
C_INT_SLAB = 480_000   # 200 mm concrete slab
C_WINDOW   = 0


In [47]:
#2 Geometry

LEN_NS  = 5.0   # N-S length, m  (width of west facade — along Barry St which runs N-S)
LEN_EW  = 4.0   # E-W depth,  m  (apartment depth — perpendicular into the building)
HEIGHT  = 2.7   # ceiling height, m

FLOOR_AREA = LEN_NS * LEN_EW                # 20.0 m²
VOLUME     = FLOOR_AREA * HEIGHT            # 54.0 m³

# Wall gross areas
A_WEST_GROSS  = LEN_NS * HEIGHT             # 13.5 m²  EXTERIOR — faces Barry St, has windows
A_EAST_GROSS  = LEN_NS * HEIGHT             # 13.5 m²  interior — to corridor
A_NORTH_GROSS = LEN_EW * HEIGHT             # 10.8 m²  interior — to Apt 306
A_SOUTH_GROSS = LEN_EW * HEIGHT             # 10.8 m²  interior — to Apt 304

# Two small horizontal-slider windows on the WEST wall (facing Barry St)
# From site photos: 2 windows side-by-side, each ~0.9 m wide × 0.9 m tall
# Only the right-hand (operable/slider) window opens; left is fixed.
WIN_WIDTH_FIXED,    WIN_HEIGHT_FIXED    = 0.9, 0.9   # fixed glazing
WIN_WIDTH_OPERABLE, WIN_HEIGHT_OPERABLE = 0.9, 0.9   # operable (horizontal slider)

A_WINDOW_FIXED    = WIN_WIDTH_FIXED * WIN_HEIGHT_FIXED         # 0.81 m²
A_WINDOW_OPERABLE = WIN_WIDTH_OPERABLE * WIN_HEIGHT_OPERABLE   # 0.81 m²
A_WINDOW_TOTAL    = A_WINDOW_FIXED + A_WINDOW_OPERABLE         # 1.62 m²

# Opaque part of the west wall (= gross − both windows)
A_WEST_OPAQUE = A_WEST_GROSS - A_WINDOW_TOTAL                  # 11.88 m²

# Sanity check: window-to-wall ratio of the west facade ≈ 12%
# 0.9 + 0.9 = 1.8 m of glazing across a 5 m wide facade = 36% width coverage
# Window heights are 0.9 m on a 2.7 m wall = 33% height coverage


In [38]:
# 4 BUILDING DICTIONARY
BUI = {
    # --------- 4a) BUILDING METADATA -----------------------
    "building": {
        "name": "Apt_305_50_Barry_St_Carlton",
        "azimuth_relative_to_true_north": 0,
        "latitude":  -37.800,
        "longitude": 144.968,
        "exposed_perimeter": 0,
        "height": HEIGHT,
        "wall_thickness": 0.20,
        "n_floors": 1,
        "building_type_class": "Residential_apartment",
        "adj_zones_present": True,
        "number_adj_zone": 5,
        "net_floor_area": FLOOR_AREA,
        "construction_class": "class_iii",
        "construction_year": "2006-today",
        "country": "Australia",
    },

    # --------- 4b) ADJACENT ZONES ---------------------------
    "adjacent_zones": [
        {   # Apt 405 — studio above
            "name": "apt_above",
            "orientation_zone": {"azimuth": 270.0},
            "area_facade_elements":    np.array([A_WEST_GROSS, A_NORTH_GROSS, A_EAST_GROSS, A_SOUTH_GROSS, FLOOR_AREA, FLOOR_AREA]),
            "typology_elements":       ["OP", "OP", "OP", "OP", "OP", "OP"],
            "transmittance_U_elements":np.array([U_EXT_WALL, U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_SLAB, U_INT_SLAB]),
            "orientation_elements":    np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
            "volume": VOLUME,
            "building_type_class": "Residential_apartment",
            "a_use": FLOOR_AREA,
        },
        {   # Apt 205 — studio below
            "name": "apt_below",
            "orientation_zone": {"azimuth": 270.0},
            "area_facade_elements":    np.array([A_WEST_GROSS, A_NORTH_GROSS, A_EAST_GROSS, A_SOUTH_GROSS, FLOOR_AREA, FLOOR_AREA]),
            "typology_elements":       ["OP", "OP", "OP", "OP", "OP", "OP"],
            "transmittance_U_elements":np.array([U_EXT_WALL, U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_SLAB, U_INT_SLAB]),
            "orientation_elements":    np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
            "volume": VOLUME,
            "building_type_class": "Residential_apartment",
            "a_use": FLOOR_AREA,
        },
        {   # Apt 306 — studio to the NORTH
            "name": "apt_north",
            "orientation_zone": {"azimuth": 0.0},
            "area_facade_elements":    np.array([A_WEST_GROSS, A_NORTH_GROSS, A_EAST_GROSS, A_SOUTH_GROSS, FLOOR_AREA, FLOOR_AREA]),
            "typology_elements":       ["OP", "OP", "OP", "OP", "OP", "OP"],
            "transmittance_U_elements":np.array([U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_SLAB, U_INT_SLAB]),
            "orientation_elements":    np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
            "volume": VOLUME,
            "building_type_class": "Residential_apartment",
            "a_use": FLOOR_AREA,
        },
        {   # Apt 304 — studio to the SOUTH
            "name": "apt_south",
            "orientation_zone": {"azimuth": 180.0},
            "area_facade_elements":    np.array([A_WEST_GROSS, A_NORTH_GROSS, A_EAST_GROSS, A_SOUTH_GROSS, FLOOR_AREA, FLOOR_AREA]),
            "typology_elements":       ["OP", "OP", "OP", "OP", "OP", "OP"],
            "transmittance_U_elements":np.array([U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_SLAB, U_INT_SLAB]),
            "orientation_elements":    np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
            "volume": VOLUME,
            "building_type_class": "Residential_apartment",
            "a_use": FLOOR_AREA,
        },
        {   # Building corridor — runs along the east side
            "name": "corridor",
            "orientation_zone": {"azimuth": 90.0},
            "area_facade_elements":    np.array([81.0, 5.4, 81.0, 5.4, 60.0, 60.0]),
            "typology_elements":       ["OP", "OP", "OP", "OP", "OP", "OP"],
            "transmittance_U_elements":np.array([U_INT_WALL] * 6),
            "orientation_elements":    np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
            "volume": 162.0,
            "building_type_class": "Residential_apartment",
            "a_use": 60.0,
        },
    ],

    # --------- 4c) ENVELOPE SURFACES ------------------------
    "building_surface": [

        # 1) WEST EXTERIOR WALL — opaque brick (Barry St facade)
        {
            "name": "West exterior wall (opaque)",
            "type": "opaque",
            "area": A_WEST_OPAQUE,
            "sky_view_factor": 0.5,
            "u_value": U_EXT_WALL,
            "solar_absorptance": ABS_EXT_WALL,
            "thermal_capacity": C_EXT_WALL,
            "orientation": {"azimuth": 270.0, "tilt": 90.0},
            "name_adj_zone": None,
            "height": HEIGHT,
            "length": LEN_NS,
        },

        # 2) NORTH INTERIOR WALL (to Apt 306)
        {
            "name": "North wall to Apt 306",
            "type": "opaque",
            "area": A_NORTH_GROSS,
            "sky_view_factor": 0.0,
            "u_value": U_INT_WALL,
            "solar_absorptance": ABS_INT,
            "thermal_capacity": C_INT_WALL,
            "orientation": {"azimuth": 0.0, "tilt": 90.0},
            "name_adj_zone": "apt_north",
            "height": HEIGHT,
            "length": LEN_EW,
        },

        # 3) SOUTH INTERIOR WALL (to Apt 304)
        {
            "name": "South wall to Apt 304",
            "type": "opaque",
            "area": A_SOUTH_GROSS,
            "sky_view_factor": 0.0,
            "u_value": U_INT_WALL,
            "solar_absorptance": ABS_INT,
            "thermal_capacity": C_INT_WALL,
            "orientation": {"azimuth": 180.0, "tilt": 90.0},
            "name_adj_zone": "apt_south",
            "height": HEIGHT,
            "length": LEN_EW,
        },

        # 4) EAST INTERIOR WALL (to corridor)
        {
            "name": "East wall to corridor",
            "type": "opaque",
            "area": A_EAST_GROSS,
            "sky_view_factor": 0.0,
            "u_value": U_INT_WALL,
            "solar_absorptance": ABS_INT,
            "thermal_capacity": C_INT_WALL,
            "orientation": {"azimuth": 90.0, "tilt": 90.0},
            "name_adj_zone": "corridor",
            "height": HEIGHT,
            "length": LEN_NS,
        },

        # 5) FLOOR (to Apt 205 below)
        {
            "name": "Floor to Apt 205",
            "type": "opaque",
            "area": FLOOR_AREA,
            "sky_view_factor": 0.0,
            "u_value": U_INT_SLAB,
            "solar_absorptance": ABS_INT,
            "thermal_capacity": C_INT_SLAB,
            "orientation": {"azimuth": 0.0, "tilt": 0.0},
            "name_adj_zone": "apt_below",
            "height": LEN_NS,
            "length": LEN_EW,
        },

        # 6) CEILING (to Apt 405 above)
        {
            "name": "Ceiling to Apt 405",
            "type": "opaque",
            "area": FLOOR_AREA,
            "sky_view_factor": 0.0,
            "u_value": U_INT_SLAB,
            "solar_absorptance": ABS_INT,
            "thermal_capacity": C_INT_SLAB,
            "orientation": {"azimuth": 0.0, "tilt": 0.0},
            "name_adj_zone": "apt_above",
            "height": LEN_NS,
            "length": LEN_EW,
        },

        # 7) WEST WINDOW — FIXED (left-hand pane, non-opening)
        {
            "name": "West window — fixed",
            "type": "transparent",
            "area": A_WINDOW_FIXED,
            "sky_view_factor": 0.5,
            "u_value": U_WINDOW,
            "solar_absorptance": 0.5,
            "thermal_capacity": C_WINDOW,
            "orientation": {"azimuth": 270.0, "tilt": 90.0},
            "name_adj_zone": None,
            "height": WIN_HEIGHT_FIXED,
            "g_value": G_WINDOW,
            "width": WIN_WIDTH_FIXED,
            "parapet": 1.0,
            "shading": True,
            "shading_type": "horizontal_overhang",
            "width_or_distance_of_shading_elements": 0.05,
            "overhang_proprieties": {
                "width_of_horizontal_overhangs": 0.25,
            },
        },

        # 8) WEST WINDOW — OPERABLE horizontal slider (right-hand pane)
        {
            "name": "West window — operable",
            "type": "transparent",
            "area": A_WINDOW_OPERABLE,
            "sky_view_factor": 0.5,
            "u_value": U_WINDOW,
            "solar_absorptance": 0.5,
            "thermal_capacity": C_WINDOW,
            "orientation": {"azimuth": 270.0, "tilt": 90.0},
            "name_adj_zone": None,
            "height": WIN_HEIGHT_OPERABLE,
            "g_value": G_WINDOW,
            "width": WIN_WIDTH_OPERABLE,
            "parapet": 1.0,
            "shading": True,
            "shading_type": "horizontal_overhang",
            "width_or_distance_of_shading_elements": 0.05,
            "overhang_proprieties": {
                "width_of_horizontal_overhangs": 0.25,
            },
        },
    ],

    # --------- 4d) UNITS DOCUMENTATION ---------------------
    "units": {
        "area": "m²",
        "u_value": "W/m²K",
        "thermal_capacity": "J/m²K",
        "azimuth": "degrees (0=N, 90=E, 180=S, 270=W)",
        "tilt": "degrees (0=horizontal, 90=vertical)",
        "internal_gain": "W/m²",
        "HVAC_profile": "0: off, 1: on",
    },

    # --------- 4e) OPERATION -------------------------------
   "building_parameters": {

        "temperature_setpoints": {
            # Heater is used "only on cold nights, very small amount"
            # → lower than standard 20 °C setpoint
            "heating_setpoint": 18.0,
            "heating_setback":  15.0,
            # No AC — windows open in summer, tolerate higher temps
            "cooling_setpoint": 26.0,
            "cooling_setback":  28.0,
            "units": "°C",
        },

        "system_capacities": {
            "heating_capacity": 10_000_000.0,
            "cooling_capacity": 10_000_000.0,
            "units": "W",
        },

         "ventilation": {
            "ventilation_type": "occupancy",
            "flow_rate_per_person": 2.0,
            "units": "l/(s m²)",
            "custom_heat_transfer_coefficient_ventilation": None,
            "info": "Annual-average rate; real summer ACH much higher (open window)",
        },

        "internal_gains": [
            {
                "name": "occupants",
                "full_load": 8.0,    # 2 ppl × ~80 W metabolic / 20 m² = 8 W/m² peak
                # Weekday: both home overnight, friend leaves 08, user mostly home,
                # both back together 20:00 onward
                #             0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18  19  20  21  22  23
                "weekday": [1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5,0.4,0.5,0.5,0.5,0.4,0.5,0.5,0.5,0.5,0.5,1.0,1.0,1.0,1.0],
                # Weekend: friend home; user sometimes out — overall ~70-80 % occupied
                "weekend": [1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.8,0.7,0.7,0.7,0.7,0.5,0.5,0.7,0.8,1.0,1.0,1.0,1.0,1.0,1.0],
            },
           {
                "name": "appliances",
                "full_load": 25.0,   # ~500 W peak in 20 m² — driven by cooking spike
                # Baseline = fridge (~50 W) + electronics when user works (~80 W).
                # Spike at 20:00 = stove + air fryer + microwave simultaneously,
                # no extractor → all heat retained. Small kettle bumps 07-08 & 22-23.
                "weekday": [0.1,0.1,0.1,0.1,0.1,0.1,0.2,0.3,0.2,0.2,0.2,0.2,0.3,0.2,0.2,0.2,0.2,0.3,0.3,0.4,1.0,0.6,0.4,0.2],
                "weekend": [0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.2,0.3,0.4,0.3,0.3,0.4,0.3,0.3,0.3,0.3,0.4,0.4,0.5,1.0,0.6,0.4,0.2],
            },
            {
                "name": "lighting",
                "full_load": 3.0,
                "weekday": [0.0,0.0,0.0,0.0,0.0,0.0,0.3,0.3,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.5,0.8,0.8,0.8,0.7,0.4,0.1],
                "weekend": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.3,0.3,0.2,0.2,0.2,0.2,0.2,0.2,0.3,0.5,0.8,0.8,0.8,0.7,0.4,0.1],
            },
        ],

        "construction": {
            "wall_thickness": 0.20,
            "thermal_bridges": 1.5,
            "units": "m (thickness), W/mK (thermal bridges)",
        },

        "climate_parameters": {
            "coldest_month": 7,
            "units": "1-12 (January-December)",
        },

        "heating_profile": {
            "weekday": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0],
            "weekend": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0],
        },
        "cooling_profile": {
            "weekday": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0],
            "weekend": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0],
        },
        "ventilation_profile": {
            "weekday": [1.0] * 24,
            "weekend": [1.0] * 24,
        },
    },
}


In [39]:
def run_iso52016(building_obj):
    kwargs = {"weather_source": WEATHER_SOURCE}
    if WEATHER_SOURCE == "epw":
        kwargs["path_weather_file"] = str(WEATHER_FILE)

    out = ISO52016.Temperature_and_Energy_needs_calculation(building_obj, **kwargs)
    if isinstance(out, tuple):
        if len(out) == 3:
            return out
        if len(out) == 2:
            return out[0], out[1], {}
    raise RuntimeError("Unexpected ISO52016 output signature")

In [48]:
print("\n=== Validating building input ===")
bui_fixed, report = sanitize_and_validate_BUI(BUI, fix=True)
for r in report:
    fix_tag = " (fix applied)" if r["fix_applied"] else ""
    print(f"  [{r['level']}] {r['path']}: {r['msg']}{fix_tag}")

errors = [r for r in report if r["level"] == "ERROR"]
if errors:
    raise ValueError("Building input invalid — fix errors above and rerun.")


=== Validating building input ===
  [WARN] building.exposed_perimeter: exposed_perimeter era 0; impostato a 1.0 (fix applied)
  [WARN] building_surface[6].parapet: parapet fuori range; clippato a 0.9 (fix applied)
  [WARN] building_surface[7].parapet: parapet fuori range; clippato a 0.9 (fix applied)


In [51]:
print(f"\n=== Running ISO 52016 simulation (weather: {WEATHER_SOURCE}) ===")
hourly_sim, annual_results_df, sankey_data = run_iso52016(bui_fixed)

OUTPUT_DIR = Path.cwd() / "result_apt305_carlton"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

hourly_sim.to_csv(OUTPUT_DIR / "hourly_simulation.csv")
annual_results_df.to_csv(OUTPUT_DIR / "annual_results.csv", index=False)

# Then continue with the rest — energy post-processing, scenarios, etc.


=== Running ISO 52016 simulation (weather: pvgis) ===


  0%|          | 8/9504 [00:00<02:09, 73.34it/s]

[t=0] T_op0=28.00  Phi_HC=-106.9  int_gains=660.1  Phi_solar=0.0  H_ve_nat=12.112
[t=1] T_op0=28.00  Phi_HC=-456.9  int_gains=660.1  Phi_solar=0.0  H_ve_nat=12.112
[t=2] T_op0=28.00  Phi_HC=-460.6  int_gains=660.1  Phi_solar=0.0  H_ve_nat=12.112
[t=3] T_op0=28.00  Phi_HC=-479.0  int_gains=660.1  Phi_solar=0.0  H_ve_nat=12.112
[t=4] T_op0=28.00  Phi_HC=-457.3  int_gains=660.1  Phi_solar=0.0  H_ve_nat=12.112
[t=5] T_op0=28.00  Phi_HC=-440.8  int_gains=660.1  Phi_solar=0.0  H_ve_nat=12.112


100%|██████████| 9504/9504 [02:03<00:00, 76.69it/s]

SANKEY CHECK  inputs=6900542.9  outputs+storage=5651725.7  residual=1248817.3 Wh (18.097%)


In [52]:
Q_HC = hourly_sim["Q_HC"]
heating_need_kWh = Q_HC[Q_HC > 0].sum() / 1000.0
cooling_need_kWh = -Q_HC[Q_HC < 0].sum() / 1000.0
total_need_kWh   = heating_need_kWh + cooling_need_kWh

heating_per_m2 = heating_need_kWh / FLOOR_AREA
cooling_per_m2 = cooling_need_kWh / FLOOR_AREA

print("\n=== Annual ENERGY NEED (ISO 52016 ideal-loads) ===")
print(f"  Heating need:  {heating_need_kWh:8.1f} kWh/yr  ({heating_per_m2:5.1f} kWh/m²/yr)")
print(f"  Cooling need:  {cooling_need_kWh:8.1f} kWh/yr  ({cooling_per_m2:5.1f} kWh/m²/yr)")
print(f"  Total need :   {total_need_kWh:8.1f} kWh/yr")

PEF_ELEC_AUS = 2.4

scenarios = {
    "AS-IS: electric panel heat, no AC": {
        "heat_eff": 1.00, "cool_eff": None,
    },
    "Electric heat + window AC (EER 2.5)": {
        "heat_eff": 1.00, "cool_eff": 2.50,
    },
    "Reverse-cycle split HP (SCOP 3.8 / EER 3.2)": {
        "heat_eff": 3.80, "cool_eff": 3.20,
    },
}

print("\n=== Delivered ELECTRICITY by HVAC scenario ===")
scenario_rows = []
for name, s in scenarios.items():
    e_heat = heating_need_kWh / s["heat_eff"]
    e_cool = (cooling_need_kWh / s["cool_eff"]) if s["cool_eff"] else 0.0
    e_total = e_heat + e_cool
    primary = e_total * PEF_ELEC_AUS

    print(f"\n  {name}")
    print(f"    Heating elec  : {e_heat:7.1f} kWh/yr")
    print(f"    Cooling elec  : {e_cool:7.1f} kWh/yr")
    print(f"    TOTAL elec    : {e_total:7.1f} kWh/yr  ({e_total/FLOOR_AREA:5.1f} kWh/m²/yr)")
    print(f"    Primary energy: {primary:7.1f} kWh/yr")

    scenario_rows.append({
        "scenario": name,
        "elec_heating_kWh": round(e_heat, 1),
        "elec_cooling_kWh": round(e_cool, 1),
        "elec_total_kWh":   round(e_total, 1),
        "elec_per_m2":      round(e_total / FLOOR_AREA, 1),
        "primary_kWh":      round(primary, 1),
    })

pd.DataFrame(scenario_rows).to_csv(OUTPUT_DIR / "scenarios.csv", index=False)



=== Annual ENERGY NEED (ISO 52016 ideal-loads) ===
  Heating need:       0.0 kWh/yr  (  0.0 kWh/m²/yr)
  Cooling need:    2572.0 kWh/yr  (128.6 kWh/m²/yr)
  Total need :     2572.0 kWh/yr

=== Delivered ELECTRICITY by HVAC scenario ===

  AS-IS: electric panel heat, no AC
    Heating elec  :     0.0 kWh/yr
    Cooling elec  :     0.0 kWh/yr
    TOTAL elec    :     0.0 kWh/yr  (  0.0 kWh/m²/yr)
    Primary energy:     0.0 kWh/yr

  Electric heat + window AC (EER 2.5)
    Heating elec  :     0.0 kWh/yr
    Cooling elec  :  1028.8 kWh/yr
    TOTAL elec    :  1028.8 kWh/yr  ( 51.4 kWh/m²/yr)
    Primary energy:  2469.1 kWh/yr

  Reverse-cycle split HP (SCOP 3.8 / EER 3.2)
    Heating elec  :     0.0 kWh/yr
    Cooling elec  :   803.8 kWh/yr
    TOTAL elec    :   803.8 kWh/yr  ( 40.2 kWh/m²/yr)
    Primary energy:  1929.0 kWh/yr


In [53]:
summary = pd.DataFrame({
    "metric": [
        "Floor area (m²)",
        "North glazing area (m²)",
        "North-wall glazing fraction (%)",
        "External wall solar absorptance",
        "Shading: brick eyebrow projection (mm)",
        "Heating need (kWh/yr)",
        "Cooling need (kWh/yr)",
        "Heating per m² (kWh/m²/yr)",
        "Cooling per m² (kWh/m²/yr)",
        "As-is electricity (resistance heat, no AC)",
        "Upgraded electricity (reverse-cycle HP)",
        "Upgraded primary energy (HP)",
    ],
    "value": [
        FLOOR_AREA,
        round(A_WINDOW_TOTAL, 2),
        round(100 * A_WINDOW_TOTAL / A_NORTH_GROSS, 1),
        ABS_EXT_WALL,
        250,
        round(heating_need_kWh, 1),
        round(cooling_need_kWh, 1),
        round(heating_per_m2, 1),
        round(cooling_per_m2, 1),
        round(scenario_rows[0]["elec_total_kWh"], 1),
        round(scenario_rows[2]["elec_total_kWh"], 1),
        round(scenario_rows[2]["primary_kWh"], 1),
    ],
})
summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
print(f"  → Summary: {OUTPUT_DIR}/summary.csv")

print("\n=== Done ===")

  → Summary: /content/pybuildinenergy_AIB/pybuildinenergy_AIB/pybuildinenergy_AIB/pyBuildingEnergy/result_apt305_carlton/summary.csv

=== Done ===


In [72]:
print("dtype     :", hourly_sim.index.dtype)
print("first 3   :", hourly_sim.index[:3].tolist())
print("last 3    :", hourly_sim.index[-3:].tolist())
print("monotonic :", hourly_sim.index.is_monotonic_increasing)
print("unique yrs:", pd.to_datetime(hourly_sim.index).year.unique().tolist())
print("len       :", len(hourly_sim))

dtype     : datetime64[ns]
first 3   : [Timestamp('2009-12-01 00:00:00'), Timestamp('2009-12-01 01:00:00'), Timestamp('2009-12-01 02:00:00')]
last 3    : [Timestamp('2009-12-31 21:00:00'), Timestamp('2009-12-31 22:00:00'), Timestamp('2009-12-31 23:00:00')]
monotonic : False
unique yrs: [2009]
len       : 9504


In [74]:
# --- 2) Clean the index, drop the warm-up, slice March -----------
sim = hourly_sim.copy()
sim.index = pd.to_datetime(sim.index)

# Each December timestamp appears twice (warm-up + main year).
# Keep the *second* occurrence — that's the converged main-year value.
sim = sim[~sim.index.duplicated(keep="last")].sort_index()

# Sanity check after cleaning
assert len(sim) == 8760, f"Expected 8760 hours after dedup, got {len(sim)}"

# Use month-only mask (TMY is stamped 2009, not 2026)
march = sim[sim.index.month == 3]
PERIOD_LABEL = "March (TMY → represents March 2026)"

print(f"\n=== {PERIOD_LABEL} ===")
print(f"  rows : {len(march)} hours  (expect 744 = 31×24)")
print(f"  from : {march.index.min()}")
print(f"  to   : {march.index.max()}")


=== March (TMY → represents March 2026) ===
  rows : 744 hours  (expect 744 = 31×24)
  from : 2009-03-01 00:00:00
  to   : 2009-03-31 23:00:00


In [75]:
# ============================================================
# MARCH 2026 — Energy consumption per HVAC scenario
# Slice hourly_sim to 01/03/2026–31/03/2026 and apply scenarios
# ============================================================
import pandas as pd

PERIOD_START = "2026-03-01 00:00"
PERIOD_END   = "2026-03-31 23:00"
PERIOD_LABEL = "01–31 March 2026"

# --- 1) Ensure hourly_sim has a DateTimeIndex ----------------
sim = hourly_sim.copy()
if not isinstance(sim.index, pd.DatetimeIndex):
    # pyBuildingEnergy returns integer-indexed hours by default.
    # Build a calendar starting 1 Jan 00:00 of the simulation year.
    SIM_YEAR = 2026                        # ← change if PVGIS TMY uses a different stamp
    sim.index = pd.date_range(
        start=f"{SIM_YEAR}-01-01 00:00",
        periods=len(sim),
        freq="h",
    )


# --- 3) Heating / cooling NEED for the window ----------------
Q_HC_m = march["Q_HC"]
heating_march_kWh = Q_HC_m[Q_HC_m > 0].sum() / 1000.0
cooling_march_kWh = -Q_HC_m[Q_HC_m < 0].sum() / 1000.0
total_march_kWh   = heating_march_kWh + cooling_march_kWh

print(f"\n  Heating need: {heating_march_kWh:8.1f} kWh")
print(f"  Cooling need: {cooling_march_kWh:8.1f} kWh")
print(f"  Total need  : {total_march_kWh:8.1f} kWh")

# --- 4) Apply the same scenarios dict, but on March only -----
print(f"\n=== Delivered ELECTRICITY by scenario — {PERIOD_LABEL} ===")
march_rows = []
for name, s in scenarios.items():
    e_heat  = heating_march_kWh / s["heat_eff"]
    e_cool  = (cooling_march_kWh / s["cool_eff"]) if s["cool_eff"] else 0.0
    e_total = e_heat + e_cool
    primary = e_total * PEF_ELEC_AUS

    print(f"\n  {name}")
    print(f"    Heating elec  : {e_heat:7.1f} kWh")
    print(f"    Cooling elec  : {e_cool:7.1f} kWh")
    print(f"    TOTAL elec    : {e_total:7.1f} kWh  ({e_total/FLOOR_AREA:5.1f} kWh/m²)")
    print(f"    Primary energy: {primary:7.1f} kWh")

    march_rows.append({
        "period":           PERIOD_LABEL,
        "scenario":         name,
        "elec_heating_kWh": round(e_heat, 1),
        "elec_cooling_kWh": round(e_cool, 1),
        "elec_total_kWh":   round(e_total, 1),
        "elec_per_m2":      round(e_total / FLOOR_AREA, 1),
        "primary_kWh":      round(primary, 1),
    })

march_df = pd.DataFrame(march_rows)
march_df.to_csv(OUTPUT_DIR / "scenarios_march_2026.csv", index=False)
print(f"\n  → Saved: {OUTPUT_DIR}/scenarios_march_2026.csv")


  Heating need:      0.0 kWh
  Cooling need:    266.6 kWh
  Total need  :    266.6 kWh

=== Delivered ELECTRICITY by scenario — 01–31 March 2026 ===

  AS-IS: electric panel heat, no AC
    Heating elec  :     0.0 kWh
    Cooling elec  :     0.0 kWh
    TOTAL elec    :     0.0 kWh  (  0.0 kWh/m²)
    Primary energy:     0.0 kWh

  Electric heat + window AC (EER 2.5)
    Heating elec  :     0.0 kWh
    Cooling elec  :   106.6 kWh
    TOTAL elec    :   106.6 kWh  (  5.3 kWh/m²)
    Primary energy:   255.9 kWh

  Reverse-cycle split HP (SCOP 3.8 / EER 3.2)
    Heating elec  :     0.0 kWh
    Cooling elec  :    83.3 kWh
    TOTAL elec    :    83.3 kWh  (  4.2 kWh/m²)
    Primary energy:   199.9 kWh


TypeError: unsupported operand type(s) for /: 'str' and 'str'